# Can Agent Simulation Catch Regressions Before an AI Agent Ships?

This notebook tests a **real implementation regression**, not just a prompt change.

We compare two versions of the same travel agent:

- **v1 — correct implementation:** the flight-selection function enforces an explicit refundable requirement.
- **v2 — regressed implementation:** the selection function accidentally ignores `refundable_required` and chooses the cheapest route-and-budget match.

The experiment uses paired synthetic multi-turn user scripts:

```text
Scenario
   ↓
Gemini generates user messages once
   ↓
Same user script
   ├──→ Agent v1 → choose_flight_v1()
   └──→ Agent v2 → choose_flight_v2()
   ↓
Deterministic regression check
```

This isolates the implementation bug from model randomness and makes the result reproducible enough for an engineering article.

In [ ]:
!pip install -q -U google-genai pandas

## 1. Authenticate Colab

In [ ]:
from google.colab import auth

auth.authenticate_user()
print("Authenticated.")

## 2. Configure Vertex AI

In [ ]:
import os

PROJECT_ID = "YOUR_PROJECT_ID"
LOCATION = "global"

MODEL_NAME = "gemini-2.5-flash"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

print("Project:", PROJECT_ID)
print("Location:", LOCATION)

In [ ]:
from google import genai
from google.genai import types

vertex_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION,
)

print("Vertex AI client created.")

## 3. Fixed flight inventory

In [ ]:
FLIGHTS = [
    {
        "flight_id": "F100",
        "from_city": "Bengaluru",
        "to_city": "Delhi",
        "price": 140,
        "refundable": True,
    },
    {
        "flight_id": "F101",
        "from_city": "Bengaluru",
        "to_city": "Delhi",
        "price": 95,
        "refundable": False,
    },
    {
        "flight_id": "F102",
        "from_city": "Bengaluru",
        "to_city": "Delhi",
        "price": 175,
        "refundable": True,
    },
    {
        "flight_id": "F200",
        "from_city": "Bengaluru",
        "to_city": "Mumbai",
        "price": 80,
        "refundable": False,
    },
    {
        "flight_id": "F201",
        "from_city": "Bengaluru",
        "to_city": "Mumbai",
        "price": 120,
        "refundable": True,
    },
]

FLIGHT_BY_ID = {
    f["flight_id"]: f
    for f in FLIGHTS
}

FLIGHTS

## 4. Correct vs regressed decision logic

This is the controlled defect.

### v1
Filters by refundability when the user requires it.

### v2
Accidentally ignores the refundability flag and picks the cheapest route-and-budget match.

In [ ]:
def choose_flight_v1(
    from_city,
    to_city,
    max_price=None,
    refundable_required=False,
):
    matches = [
        f for f in FLIGHTS
        if f["from_city"].lower() == from_city.lower()
        and f["to_city"].lower() == to_city.lower()
    ]

    if max_price is not None:
        matches = [
            f for f in matches
            if f["price"] <= max_price
        ]

    if refundable_required:
        matches = [
            f for f in matches
            if f["refundable"] is True
        ]

    if not matches:
        return None

    return min(
        matches,
        key=lambda f: f["price"]
    )


def choose_flight_v2(
    from_city,
    to_city,
    max_price=None,
    refundable_required=False,
):
    matches = [
        f for f in FLIGHTS
        if f["from_city"].lower() == from_city.lower()
        and f["to_city"].lower() == to_city.lower()
    ]

    if max_price is not None:
        matches = [
            f for f in matches
            if f["price"] <= max_price
        ]

    # REGRESSION:
    # refundable_required is accidentally ignored.

    if not matches:
        return None

    return min(
        matches,
        key=lambda f: f["price"]
    )


print("v1/v2 decision functions ready.")

## 5. Unit sanity check

Before involving the LLM, prove the implementation regression exists.

In [ ]:
print(
    "v1:",
    choose_flight_v1(
        "Bengaluru",
        "Delhi",
        max_price=160,
        refundable_required=True,
    )
)

print(
    "v2:",
    choose_flight_v2(
        "Bengaluru",
        "Delhi",
        max_price=160,
        refundable_required=True,
    )
)

Expected:

```text
v1 → F100 ($140, refundable)
v2 → F101 ($95, non-refundable)
```

If this sanity check does not show that difference, stop and fix the implementation before continuing.

## 6. Scenario specifications

The synthetic user generator creates the user-side wording once.

We then replay exactly the same script against both versions.

In [ ]:
SCENARIOS = [
    {
        "scenario_id": "S1",
        "goal": "Book Bengaluru to Delhi under $160 and require refundable.",
        "turn_plan": [
            "Ask for a Bengaluru to Delhi flight under $160.",
            "Then clarify that it must be refundable and ask for the best recommendation."
        ],
        "expected_regression": True,
    },
    {
        "scenario_id": "S2",
        "goal": "Find the cheapest Bengaluru to Delhi flight. Refundability does not matter.",
        "turn_plan": [
            "Ask for the cheapest Bengaluru to Delhi flight."
        ],
        "expected_regression": False,
    },
    {
        "scenario_id": "S3",
        "goal": "Book Bengaluru to Mumbai under $130 and require refundable.",
        "turn_plan": [
            "Ask for Bengaluru to Mumbai under $130.",
            "Then say it must be refundable and ask which one to choose."
        ],
        "expected_regression": True,
    },
    {
        "scenario_id": "S4",
        "goal": "Find Bengaluru to Delhi under $100 and require refundable without increasing budget.",
        "turn_plan": [
            "Ask for Bengaluru to Delhi under $100 and require refundable.",
            "If there is no valid option, insist the budget cannot increase."
        ],
        "expected_regression": True,
    },
    {
        "scenario_id": "S5",
        "goal": "Start with cheapest Bengaluru to Delhi, then add that final choice must be refundable.",
        "turn_plan": [
            "Ask for the cheapest Bengaluru to Delhi option.",
            "Then clarify that the final recommendation must be refundable."
        ],
        "expected_regression": True,
    },
    {
        "scenario_id": "S6",
        "goal": "Find any Bengaluru to Mumbai flight under $150. Refundability is not required.",
        "turn_plan": [
            "Ask for a Bengaluru to Mumbai flight under $150 and request a recommendation."
        ],
        "expected_regression": False,
    },
    {
        "scenario_id": "S7",
        "goal": "Ask for Bengaluru to Delhi under $180 and reject all non-refundable options.",
        "turn_plan": [
            "Ask for Bengaluru to Delhi under $180.",
            "Then explicitly say non-refundable flights are unacceptable and ask for the best choice."
        ],
        "expected_regression": True,
    },
    {
        "scenario_id": "S8",
        "goal": "Ask for cheapest Bengaluru to Mumbai, then request a refundable alternative under $130.",
        "turn_plan": [
            "Ask for the cheapest Bengaluru to Mumbai flight.",
            "Then ask for a refundable alternative under $130."
        ],
        "expected_regression": True,
    },
]

print(f"{len(SCENARIOS)} scenarios ready.")

## 7. Generate paired synthetic user scripts

In [ ]:
import json
import re
import pandas as pd

def generate_user_script(scenario):
    prompt = f'''
You are creating a short user-only script for an AI travel-agent regression test.

GOAL:
{scenario["goal"]}

TURN PLAN:
{json.dumps(scenario["turn_plan"], indent=2)}

Create exactly {len(scenario["turn_plan"])} realistic USER messages.

Requirements:
- Preserve every constraint in the turn plan.
- If a constraint is supposed to appear on a later turn, do not reveal it early.
- Keep messages concise and natural.
- Do not include assistant messages.
- Return JSON only in this shape:
  {{"messages": ["message 1", "message 2"]}}
'''

    response = vertex_client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json"
        ),
    )

    data = json.loads(response.text)
    messages = data["messages"]

    if len(messages) != len(scenario["turn_plan"]):
        raise ValueError(
            f'{scenario["scenario_id"]}: expected '
            f'{len(scenario["turn_plan"])} messages, got {len(messages)}'
        )

    return messages


USER_SCRIPTS = {}

for scenario in SCENARIOS:
    sid = scenario["scenario_id"]
    print("Generating:", sid)
    USER_SCRIPTS[sid] = generate_user_script(scenario)

print("Paired user scripts generated.")

In [ ]:
for sid, messages in USER_SCRIPTS.items():
    print("=" * 70)
    print(sid)
    for i, message in enumerate(messages, 1):
        print(f"Turn {i}: {message}")

## 8. Constraint extraction

The agent uses Gemini to turn the latest conversation into structured constraints.

This simulates a common production pattern:

```text
natural-language conversation
        ↓
LLM extracts structured intent
        ↓
business logic / tool executes
```

In [ ]:
def render_conversation(messages):
    return "\n".join(
        f'{m["role"].upper()}: {m["content"]}'
        for m in messages
    )


def extract_constraints(conversation):
    prompt = f'''
Extract the user's current travel constraints from this conversation.

CONVERSATION:
{render_conversation(conversation)}

Return JSON only with exactly these fields:
{{
  "from_city": string or null,
  "to_city": string or null,
  "max_price": number or null,
  "refundable_required": true or false
}}

Rules:
- Use the latest user intent.
- Preserve constraints stated in earlier turns unless the user changes them.
- refundable_required must be true if the user explicitly requires refundable,
  rejects non-refundable flights, or asks for a refundable alternative.
'''

    response = vertex_client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json"
        ),
    )

    return json.loads(response.text)

## 9. Agent response helper

The LLM explains the result, but the actual selection comes from the deterministic decision function.

That is important: the behavioral regression lives in business logic, not in prompt wording.

In [ ]:
def build_agent_response(
    conversation,
    constraints,
    selected_flight,
):
    prompt = f'''
You are a travel assistant.

USER CONVERSATION:
{render_conversation(conversation)}

EXTRACTED CONSTRAINTS:
{json.dumps(constraints, indent=2)}

DECISION-LAYER RESULT:
{json.dumps(selected_flight, indent=2)}

Respond naturally to the user.

Rules:
- Do not invent another flight.
- If DECISION-LAYER RESULT is null, say there is no matching option.
- If it is not null, recommend that exact flight.
- End with exactly one final line:
  RECOMMENDATION: <flight_id>
  or
  RECOMMENDATION: NONE
'''

    response = vertex_client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
    )

    return response.text.strip()

## 10. Recommendation parser

In [ ]:
RECOMMENDATION_PATTERN = re.compile(
    r"RECOMMENDATION:\s*(F\d{3}|NONE)\s*$",
    re.IGNORECASE | re.MULTILINE,
)


def extract_recommendation(text):
    matches = RECOMMENDATION_PATTERN.findall(text or "")

    if not matches:
        return None

    value = matches[-1].upper()

    if value == "NONE":
        return None

    return value if value in FLIGHT_BY_ID else None

## 11. Run one conversation against a decision function

In [ ]:
def run_conversation(
    scenario,
    user_messages,
    decision_fn,
    version_name,
):
    conversation = []
    turns = []

    for turn_number, user_message in enumerate(user_messages, 1):
        conversation.append({
            "role": "user",
            "content": user_message,
        })

        constraints = extract_constraints(conversation)

        selected = None

        if (
            constraints.get("from_city")
            and constraints.get("to_city")
        ):
            selected = decision_fn(
                from_city=constraints["from_city"],
                to_city=constraints["to_city"],
                max_price=constraints.get("max_price"),
                refundable_required=constraints.get(
                    "refundable_required",
                    False
                ),
            )

        assistant_message = build_agent_response(
            conversation=conversation,
            constraints=constraints,
            selected_flight=selected,
        )

        conversation.append({
            "role": "assistant",
            "content": assistant_message,
        })

        turns.append({
            "turn": turn_number,
            "user_message": user_message,
            "constraints": constraints,
            "selected_flight": selected,
            "recommendation": extract_recommendation(
                assistant_message
            ),
            "assistant_message": assistant_message,
        })

    return {
        "scenario_id": scenario["scenario_id"],
        "version": version_name,
        "goal": scenario["goal"],
        "expected_regression": scenario["expected_regression"],
        "conversation": conversation,
        "turns": turns,
    }

## 12. Deterministic evaluator

We score the decision-layer output, not arbitrary text mentions.

A violation occurs when:

- `refundable_required == True`, and
- the selected flight is non-refundable.

In [ ]:
def score_result(result):
    violations = []

    for turn in result["turns"]:
        constraints = turn["constraints"]
        selected = turn["selected_flight"]

        if not constraints.get(
            "refundable_required",
            False
        ):
            continue

        if selected is None:
            continue

        if selected["refundable"] is False:
            violations.append({
                "turn": turn["turn"],
                "flight_id": selected["flight_id"],
                "constraints": constraints,
            })

    final_selected = None

    if result["turns"]:
        final_selected = result["turns"][-1]["selected_flight"]

    return {
        **result,
        "final_selected_flight": (
            final_selected["flight_id"]
            if final_selected
            else None
        ),
        "refundable_violation": len(violations) > 0,
        "violation_count": len(violations),
        "violation_details": violations,
    }

## 13. Run v1

In [ ]:
results_v1 = []

for scenario in SCENARIOS:
    sid = scenario["scenario_id"]
    print("Running v1:", sid)

    result = run_conversation(
        scenario=scenario,
        user_messages=USER_SCRIPTS[sid],
        decision_fn=choose_flight_v1,
        version_name="v1",
    )

    results_v1.append(
        score_result(result)
    )

print("v1 complete.")

## 14. Run v2

In [ ]:
results_v2 = []

for scenario in SCENARIOS:
    sid = scenario["scenario_id"]
    print("Running v2:", sid)

    result = run_conversation(
        scenario=scenario,
        user_messages=USER_SCRIPTS[sid],
        decision_fn=choose_flight_v2,
        version_name="v2",
    )

    results_v2.append(
        score_result(result)
    )

print("v2 complete.")

## 15. Compare versions

In [ ]:
def summarize(results):
    return pd.DataFrame([
        {
            "scenario_id": r["scenario_id"],
            "version": r["version"],
            "expected_regression": r["expected_regression"],
            "final_selected_flight": r["final_selected_flight"],
            "refundable_violation": r["refundable_violation"],
            "violation_count": r["violation_count"],
        }
        for r in results
    ])


comparison = pd.concat(
    [
        summarize(results_v1),
        summarize(results_v2),
    ],
    ignore_index=True,
)

comparison

In [ ]:
summary = (
    comparison
    .groupby("version")
    .agg(
        scenarios=("scenario_id", "count"),
        refundable_violations=("refundable_violation", "sum"),
        total_violation_turns=("violation_count", "sum"),
    )
    .reset_index()
)

summary

## 16. Paired regression detection

A scenario is counted as caught when:

- v1 does not violate the refundable constraint;
- v2 does.

In [ ]:
v1_by_id = {
    r["scenario_id"]: r
    for r in results_v1
}

v2_by_id = {
    r["scenario_id"]: r
    for r in results_v2
}

caught_regressions = []

for scenario in SCENARIOS:
    sid = scenario["scenario_id"]

    v1 = v1_by_id[sid]
    v2 = v2_by_id[sid]

    if (
        not v1["refundable_violation"]
        and v2["refundable_violation"]
    ):
        caught_regressions.append(sid)

print("Regression scenarios caught:", caught_regressions)
print("Count:", len(caught_regressions))

## 17. Expected-vs-observed check

This helps detect whether the synthetic scripts actually exercised the intended regression cases.

In [ ]:
expected_regression_ids = {
    s["scenario_id"]
    for s in SCENARIOS
    if s["expected_regression"]
}

caught_set = set(caught_regressions)

print("Expected regression scenarios:", sorted(expected_regression_ids))
print("Observed caught scenarios:", sorted(caught_set))
print("Missed expected scenarios:", sorted(expected_regression_ids - caught_set))
print("Unexpected regression scenarios:", sorted(caught_set - expected_regression_ids))

## 18. Inspect caught regressions

In [ ]:
def print_result(result):
    print("=" * 90)
    print("Scenario:", result["scenario_id"])
    print("Version:", result["version"])
    print("Goal:", result["goal"])
    print("Final selected:", result["final_selected_flight"])
    print("Violation:", result["refundable_violation"])
    print("-" * 90)

    for turn in result["turns"]:
        print("USER:", turn["user_message"])
        print("CONSTRAINTS:", turn["constraints"])
        print("SELECTED:", turn["selected_flight"])
        print("ASSISTANT:", turn["assistant_message"])
        print()


for sid in caught_regressions:
    print("\nBASELINE")
    print_result(v1_by_id[sid])

    print("\nREGRESSED")
    print_result(v2_by_id[sid])

## 19. Manual smoke tests

These deliberately avoid refundability.

The point is to show that ordinary route/price smoke tests can pass both implementations even while multi-turn simulated users expose the regression.

In [ ]:
MANUAL_CASES = [
    {
        "name": "Delhi cheapest",
        "from_city": "Bengaluru",
        "to_city": "Delhi",
        "max_price": None,
        "refundable_required": False,
    },
    {
        "name": "Delhi under 200",
        "from_city": "Bengaluru",
        "to_city": "Delhi",
        "max_price": 200,
        "refundable_required": False,
    },
    {
        "name": "Mumbai under 150",
        "from_city": "Bengaluru",
        "to_city": "Mumbai",
        "max_price": 150,
        "refundable_required": False,
    },
    {
        "name": "Mumbai cheapest",
        "from_city": "Bengaluru",
        "to_city": "Mumbai",
        "max_price": None,
        "refundable_required": False,
    },
    {
        "name": "Delhi under 100",
        "from_city": "Bengaluru",
        "to_city": "Delhi",
        "max_price": 100,
        "refundable_required": False,
    },
]

manual_rows = []

for case in MANUAL_CASES:
    v1 = choose_flight_v1(**{
        k: case[k]
        for k in [
            "from_city",
            "to_city",
            "max_price",
            "refundable_required",
        ]
    })

    v2 = choose_flight_v2(**{
        k: case[k]
        for k in [
            "from_city",
            "to_city",
            "max_price",
            "refundable_required",
        ]
    })

    manual_rows.append({
        "case": case["name"],
        "v1": v1["flight_id"] if v1 else None,
        "v2": v2["flight_id"] if v2 else None,
        "same": (
            (v1["flight_id"] if v1 else None)
            ==
            (v2["flight_id"] if v2 else None)
        ),
    })

manual_df = pd.DataFrame(manual_rows)
manual_df

## 20. Final experiment facts

In [ ]:
manual_same_count = int(
    manual_df["same"].sum()
)

v1_violation_scenarios = int(
    summary.loc[
        summary["version"] == "v1",
        "refundable_violations"
    ].iloc[0]
)

v2_violation_scenarios = int(
    summary.loc[
        summary["version"] == "v2",
        "refundable_violations"
    ].iloc[0]
)

print("=== EXPERIMENT FACTS ===")
print(
    f"Manual smoke tests with same output: "
    f"{manual_same_count}/{len(MANUAL_CASES)}"
)
print(
    "v1 refundable-violation scenarios:",
    v1_violation_scenarios
)
print(
    "v2 refundable-violation scenarios:",
    v2_violation_scenarios
)
print(
    "Paired regressions caught:",
    len(caught_regressions)
)
print(
    "Scenario IDs:",
    caught_regressions
)